# Exp3 evaluation: LLM-agent investigations

A correct process does not guarantee a correct result (lecture 12), so the agent is scored on both.

| Level | Metric | Question | How it is measured |
|---|---|---|---|
| Process | Tool recall | Did it check every data source? | sources checked without error / all sources (needs `TOOL_MAP`) |
| Process | Argument accuracy | Did it look up the right applicant? | tool calls with this applicant's `SK_ID_CURR` / calls passing an `SK_ID_CURR` |
| Process | Gate adherence | Did high-risk or incomplete cases go to a human? | escalated / runs that had to be (rating `high` or a failed tool call); must be 100% |
| Process | Faithfulness | Are the agent's claims true to the data? | share of cited values that match what its tools returned |
| Process | Completeness | Does it find the red flags that are present? | share of the applicant's rule-based red flags (`red_flags.parquet`) it cited (needs `FLAG_MAP`) |
| Process | Uncertainty honesty | Does it say "no history" instead of "clean"? | sources with no records reported as `no_history`/`unknown`; failed checks as `unknown` |
| Process | Consistency | Same answer when run again? | rating agreement, risk-score spread, overlap of cited evidence across runs |
| Output | Judge rubric | Is the written report grounded, hedged and complete? | LLM judge scores 0-2 (grounding is blocking), checked against human grades |
| Outcome | **Discrimination in the flagged group (main)** | Among applicants the Exp1 model flagged, does the agent separate real defaulters from false alarms? | PR-AUC / ROC-AUC of the risk score and precision / recall of the `high` rating, vs the group's default rate (= precision of the Exp1 flag) and, paired, vs the Exp1 score |
| Outcome | Over-flagging (control group) | Does it rate ordinary applicants high? | share of random controls rated `high`; false-positive rate |
| Outcome | Discrimination, whole cohort (secondary) | | same metrics on flagged + control |
| Outcome | Calibration | Is a risk score of 0.8 right 80% of the time? | Brier score, expected calibration error, reliability table |
| Outcome | Fairness | Are some groups flagged more for the same risk? | flag rate and false-positive rate by gender / age band (audit only) |
| Outcome | Cost | What does one investigation cost? | tokens and latency per run |

**Why the flagged group is the main result.** The cohort is the top 100 by the Exp1 model plus 100 random controls.
It was chosen with the Exp1 score, so on the whole cohort Exp1 has a built-in head start (ROC-AUC 0.84 there, but
0.65 inside the top 100), and the default rate (about 29%) is far above the 8% of Exp1/Exp2. The agent's job is to
investigate applicants the model has already flagged, so the fair test is inside the flagged group: about half of
them default, and the question is whether the agent's rating picks out which half. With ~100 applicants the CIs are
wide; only a large improvement will be significant.

**Why completeness uses rule-based red flags, not SHAP.** The answer key does not depend on the Exp1 model, so the
agent cannot score well just by repeating the model's reasoning.

**How to use:** (1) build the cohort, (2) map red flags and data sources to your tool names, (3) run the agent and
point `RUNS` at its output, (4) optionally grade the reports with the judge, then **Run All**. If `RUNS` is empty, a
demo with made-up runs is shown.

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "outputs" / "exp3"] if (p / "eval_exp3.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp3 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Cohort: who to investigate
Test applicants only (the Exp1 model never saw them): the riskiest by the Exp1 model plus a random control group.
The controls are needed to measure discrimination. Set `COHORT_FROM` to the Exp1 test scores (CSV path or DataFrame
with `SK_ID_CURR, score`).

In [2]:
COHORT_FROM = None      # e.g. "path/to/exp1_test_scores.csv"
K_FLAGGED, K_CONTROL = 100, 100

if COHORT_FROM is not None:
    cohort = ev.select_cohort(COHORT_FROM, K_FLAGGED, K_CONTROL)
    cohort.to_csv(HERE / "cohort.csv", index=False)
    print(f"Saved cohort.csv: {len(cohort)} applicants")
    display(cohort.group.value_counts().to_frame("applicants"))
else:
    print("COHORT_FROM not set - skipping.")

COHORT_FROM not set - skipping.


## 2. Red flags, data sources and your tools
* **`FLAG_MAP`** (completeness): the answer key has 21 rules. Each rule needs the field name(s) **your agent's tools**
  return as evidence for it (e.g. `INST_FREQUENT_LATE` -> `["late_rate"]`). Rules you leave out can never be "found".
* **`TOOL_MAP`** (tool recall, failed-check honesty): which of your tools reads each data source. The example names
  in `ev.TOOL_MAP_TEMPLATE` must be replaced with your own.

In [3]:
rules = pd.read_json(HERE / "red_flag_rules.json").set_index("id")
rules

,source_table,concept,rule,threshold
id,,,,
HIGH_ANNUITY_TO_INCOME,application,repayment burden,annuity / income in top 5% (train),0.354033
LOW_EXT_SCORE,application,external credit score,mean EXT_SOURCE in bottom 10% (train),0.304384
RECENT_ID_CHANGE,application,identity change,ID document changed within 1 year,NaN
RECENT_PHONE_CHANGE,application,identity change,phone changed within 30 days,NaN
SHORT_EMPLOYMENT,application,employment,employed less than 1 year,NaN
MANY_ENQUIRIES,application,credit enquiries,bureau enquiries last year in top 5% (train),6.000000
BUREAU_CURRENT_OVERDUE,bureau,current overdue debt,overdue amount at another lender > 0,NaN
BUREAU_SEVERE_DELINQUENCY,bureau_balance,past delinquency,worst monthly status 31+ days past due,NaN
BUREAU_HIGH_UTILISATION,bureau,debt burden,active debt >= 80% of active credit,NaN


In [4]:
FLAG_MAP = {
    # "INST_FREQUENT_LATE": ["late_rate"],
    # "PREV_HIGH_REFUSAL": ["refusal_rate"],
}   # or a path to a JSON file with the same structure

TOOL_MAP = {
    # "installments": ["get_installment_history"],      # see ev.TOOL_MAP_TEMPLATE for all six sources
}   # or a path to a JSON file with the same structure

## 3. Agent runs
A list of run dicts, or a `.jsonl` file with one JSON object per line, for example:
```json
{"SK_ID_CURR": 100054, "run": 1, "risk_score": 0.82, "rating": "high", "escalated": true,
 "citations": [{"tool": "get_installment_history", "field": "late_rate", "value": 0.21}],
 "tool_outputs": {"get_installment_history": {"late_rate": 0.214, "max_days_late": 41}},
 "tool_calls": [{"tool": "get_installment_history", "args": {"SK_ID_CURR": 100054}, "status": "ok"}],
 "source_findings": {"installments": "concern", "credit_card": "no_history"},
 "usage": {"input_tokens": 5200, "output_tokens": 640, "latency_s": 14.2},
 "report": "The applicant paid 21% of instalments late ..."}
```
* `tool_calls[].status`: `ok`, `no_history` (the tool ran and found no records) or `error` (the check did not complete).
* `source_findings`: the agent's verdict per data source (`application`, `bureau`, `previous_application`,
  `installments`, `pos_cash`, `credit_card`): `concern`, `clean`, `no_history` or `unknown`.
* Everything after `tool_outputs` is optional; metrics that need a missing field are left out.
* Cite values exactly as the tools return them (fractions such as `0.214`, not "21%"), or faithfulness marks them wrong.

Run each applicant several times (e.g. 3) to measure consistency. Never give the agent `CODE_GENDER` or
`NAME_FAMILY_STATUS`; they are used only for the fairness audit below.

In [5]:
RUNS = None              # list of dicts or path to agent_runs.jsonl
COHORT = None            # cohort from section 1 (DataFrame or CSV); None = cohort.csv if it exists
BASELINE = None          # optional: other model scores (CSV/DataFrame); default = the Exp1 score stored in the cohort
FLAG_RATINGS = ("high",) # ratings that count as flagged and must be escalated to a human
JUDGE = None             # optional: LLM-judge rubric scores (CSV path or DataFrame), see section 5
HUMAN = None             # optional: human rubric scores for some of the same reports
N_BOOT = 1000
SAVE = True

In [6]:
DEMO = RUNS is None
if DEMO:
    import numpy as np
    rng = np.random.default_rng(42)
    sp = ev.load_splits()
    test = sp[sp.split == "test"]
    ids = list(test[test.TARGET == 1].SK_ID_CURR.head(6)) + list(test[test.TARGET == 0].SK_ID_CURR.head(6))
    has = ev.source_coverage(ids)
    TOOL_MAP = TOOL_MAP or ev.TOOL_MAP_TEMPLATE
    RUNS, JUDGE = [], []
    for k, sid in enumerate(ids):
        tools = {"get_installment_history": {"late_rate": round(0.3 - 0.05 * k, 3), "max_days_late": 40 - 5 * k},
                 "get_previous_applications": {"refusal_rate": 0.5 if k < 3 else 0.0}}
        for run in range(1, 4):
            cites = [{"tool": "get_installment_history", "field": "late_rate", "value": tools["get_installment_history"]["late_rate"]},
                     {"tool": "get_installment_history", "field": "max_days_late", "value": 40 - 5 * k if run < 3 else 99}]
            calls = [{"tool": t[0], "args": {"SK_ID_CURR": int(sid)}, "status": "ok" if has.at[sid, s] else "no_history"}
                     for s, t in TOOL_MAP.items() if not (k == 4 and s == "bureau")]          # applicant 5 skips the bureau
            findings = {s: ("concern" if has.at[sid, s] and k < 3 else "clean" if has.at[sid, s] or k == 5 else "no_history")
                        for s in TOOL_MAP}                                                      # applicant 6 says 'clean' for no data
            rating = "high" if k < 4 or k == 9 else "low"
            RUNS.append({"SK_ID_CURR": int(sid), "run": run, "risk_score": float(np.clip(0.8 - 0.05 * k + rng.normal(0, 0.03), 0, 1)),
                         "rating": rating, "escalated": rating == "high" and k != 2,            # applicant 3 is never escalated
                         "citations": cites, "tool_outputs": tools, "tool_calls": calls, "source_findings": findings,
                         "usage": {"input_tokens": 5000 + 100 * k, "output_tokens": 600, "latency_s": 12.0 + k}})
            JUDGE.append({"SK_ID_CURR": int(sid), "run": run, "grounding": 2 if run < 3 else 0, "hedging": 2 if k != 5 else 0,
                          "completeness": 2 if k != 4 else 1})
    JUDGE = pd.DataFrame(JUDGE)
    COHORT = pd.DataFrame({"SK_ID_CURR": ids, "score": rng.uniform(0.1, 0.6, len(ids)),
                           "group": ["flagged" if k % 6 < 4 else "control" for k in range(len(ids))]})
    HUMAN = JUDGE[JUDGE.run == 1].assign(grounding=lambda d: d.grounding.where(d.index % 2 == 0, 1))
    FLAG_MAP = FLAG_MAP or {"INST_FREQUENT_LATE": ["late_rate"], "INST_SEVERE_LATE": ["max_days_late"],
                            "PREV_HIGH_REFUSAL": ["refusal_rate"]}
    print("RUNS was empty, so this is a DEMO with made-up agent runs. These numbers are NOT results.")

RUNS was empty, so this is a DEMO with made-up agent runs. These numbers are NOT results.


## 4. Results

In [7]:
per_app, summary = ev.evaluate_agent_runs(RUNS, FLAG_MAP or None, TOOL_MAP or None, FLAG_RATINGS, BASELINE,
                                          JUDGE, HUMAN, N_BOOT, COHORT)
ev.summary_table(summary)

,value
metric,
cohort: applicants,12
cohort: runs,36
cohort: default_rate,0.500
cohort: flagged_share,0.417
process: tool_recall,0.986
process: argument_accuracy,1.000
process: gate_adherence,0.800
process: faithfulness,0.833
process: citations_not_found_share,0.000


### Per applicant
One row per investigated applicant (averaged over runs). `missed_flags` lists red flags that no run cited;
`gate_escalated_runs` < `gate_required_runs` means a high-risk or incomplete case was not sent to a human.

In [8]:
per_app.drop(columns=["is_dev"], errors="ignore")

,SK_ID_CURR,n_runs,risk_score,rating,flagged,faithfulness,n_red_flags,completeness,missed_flags,tool_recall,argument_accuracy,gate_required_runs,gate_escalated_runs,uncertainty_honesty,input_tokens,output_tokens,latency_s,same_rating_all_runs,rating_agreement,risk_score_std,risk_score_range,evidence_jaccard,TARGET,split,group,baseline_score
0,100047,3,0.800152,high,1,0.833333,0,NaN,,1.000000,1.0,3,3,NaN,5000.0,600.0,12.0,True,1.0,0.022831,0.053713,1.0,1,test,flagged,0.164961
1,100192,3,0.726874,high,1,0.833333,2,0.000000,"LOW_EXT_SCORE, SHORT_EMPLOYMENT",1.000000,1.0,3,3,1.0,5100.0,600.0,13.0,True,1.0,0.037165,0.086748,1.0,1,test,flagged,0.337852
2,100439,3,0.697948,high,1,0.833333,4,0.250000,"BUREAU_HIGH_UTILISATION, INST_WORSENING, LOW_E...",1.000000,1.0,3,0,1.0,5200.0,600.0,14.0,True,1.0,0.005548,0.013322,1.0,1,test,flagged,0.213455
3,100879,3,0.658041,high,1,0.833333,3,0.333333,"INST_WORSENING, PREV_HIGH_REFUSAL",1.000000,1.0,3,3,1.0,5300.0,600.0,15.0,True,1.0,0.023815,0.051973,1.0,1,test,flagged,0.434907
4,101564,3,0.616608,low,0,0.833333,2,0.000000,"LOW_EXT_SCORE, RECENT_PHONE_CHANGE",0.833333,1.0,0,0,1.0,5400.0,600.0,16.0,True,1.0,0.013125,0.031836,1.0,1,test,control,0.318576
5,101874,3,0.535506,low,0,0.833333,5,0.200000,"BUREAU_SEVERE_DELINQUENCY, LOW_EXT_SCORE, PREV...",1.000000,1.0,0,0,0.0,5500.0,600.0,17.0,True,1.0,0.018112,0.039829,1.0,1,test,control,0.516339
6,100026,3,0.506437,low,0,0.833333,1,0.000000,BUREAU_HIGH_UTILISATION,1.000000,1.0,0,0,1.0,5600.0,600.0,18.0,True,1.0,0.014180,0.031899,1.0,0,test,flagged,0.450133
7,100027,3,0.453871,low,0,0.833333,1,0.000000,BUREAU_HIGH_UTILISATION,1.000000,1.0,0,0,1.0,5700.0,600.0,19.0,True,1.0,0.024076,0.057104,1.0,0,test,flagged,0.256183
8,100029,3,0.397518,low,0,0.833333,2,0.000000,"BUREAU_HIGH_UTILISATION, RECENT_PHONE_CHANGE",1.000000,1.0,0,0,1.0,5800.0,600.0,20.0,True,1.0,0.013080,0.028819,1.0,0,test,flagged,0.516130
9,100056,3,0.362090,high,1,0.833333,1,1.000000,,1.000000,1.0,3,3,1.0,5900.0,600.0,21.0,True,1.0,0.000827,0.001961,1.0,0,test,flagged,0.502382


### Calibration
Equal-size bins of the agent's risk score. If `mean_risk_score` and `default_rate` differ a lot, treat the score as a
ranking, not a probability. With ~200 applicants (enriched with high-risk ones) the bins are noisy: treat this as descriptive only.

In [9]:
if per_app.TARGET.nunique() == 2 and per_app.risk_score.between(0, 1).all():
    calib = ev.calibration(per_app.TARGET, per_app.risk_score, n_bins=min(5, len(per_app) // 2))[1]
    display(calib)
else:
    calib = None
    print("Needs both defaulters and non-defaulters, and risk scores between 0 and 1.")

,n,mean_risk_score,default_rate
bin,,,
"(0.258, 0.369]",3,0.311211,0.0
"(0.369, 0.475]",2,0.425695,0.0
"(0.475, 0.584]",2,0.520971,0.5
"(0.584, 0.69]",2,0.637325,1.0
"(0.69, 0.8]",3,0.741658,1.0


### Fairness audit
Flag rate and false-positive rate (non-defaulters rated `high`) per group. Large gaps in false-positive rate between
groups are a fairness problem, not just tuning. With ~200 applicants this is descriptive: report `n` alongside. Family status is left out (its groups are too small);
for a reliable fairness check, run the same false-positive comparison on the Exp1 model's 15,000 test applicants.

In [10]:
fair = ev.fairness_table(per_app)
fair.style.format({"default_rate": "{:.1%}", "flag_rate": "{:.1%}", "false_positive_rate": "{:.1%}"})

,grouping,group,n,default_rate,flag_rate,false_positive_rate
0,CODE_GENDER,F,6,50.0%,33.3%,0.0%
1,CODE_GENDER,M,6,50.0%,50.0%,33.3%
2,AGE_BAND,30-44,3,0.0%,0.0%,0.0%
3,AGE_BAND,45-59,4,75.0%,75.0%,100.0%
4,AGE_BAND,60+,1,0.0%,0.0%,0.0%
5,AGE_BAND,<30,4,75.0%,50.0%,0.0%


## 5. LLM-as-judge (output quality)
Grade each report with a second LLM using `ev.JUDGE_RUBRIC` (`ev.judge_prompt(run)` builds the full prompt from a run's
tool outputs and `report`). Save the scores as `SK_ID_CURR, run, grounding, hedging, completeness` and set `JUDGE`.
The judge is a tool, not ground truth: have a teammate grade ~20 of the same reports blind and set `HUMAN`; the
summary then shows agreement and Cohen's kappa per criterion.

In [11]:
print(ev.judge_prompt(RUNS[0])[:1500])

You are reviewing a credit-risk investigation report written by an AI agent.
Score the report on each criterion below, using ONLY the tool outputs provided. Quote the evidence for each score.
Reply as JSON: {"grounding": 0-2, "hedging": 0-2, "completeness": 0-2, "evidence": {"grounding": "...", ...}}

GROUNDING (blocking)
  2 = every factual claim traces to a value in the tool outputs
  1 = one claim is unsourced but plausible
  0 = any claim contradicts the tool outputs, or cites a tool or value that does not exist
HEDGING
  2 = missing or failed data is stated as missing/unknown (e.g. "no credit-card history"), never as a clean record
  1 = hedged vaguely ("may have issues") without saying what is missing
  0 = states something as fact that the tool outputs do not show
COMPLETENESS
  2 = every data source has a verdict, including negative results
  1 = one data source has no verdict
  0 = two or more data sources, or a clear red flag in the tool outputs, are not addressed
Any 0 on GR

## 6. Save

In [12]:
if SAVE and not DEMO:
    import json
    per_app.to_csv(HERE / "exp3_per_applicant.csv", index=False)
    (HERE / "exp3_summary.json").write_text(json.dumps(summary, indent=2))
    fair.to_csv(HERE / "exp3_fairness.csv", index=False)
    if calib is not None:
        calib.to_csv(HERE / "exp3_calibration.csv")
    print("Saved result tables in", HERE)
else:
    print("Not saved (demo mode or SAVE = False).")

Not saved (demo mode or SAVE = False).
